In [26]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import geopandas as gpd
import os
import pyogrio

### AWS credentials not working so accessing theough ArcGIS Item

In [10]:

item_id = "b3489960c0e942c6985d2eca471718dd"

url = f"https://www.arcgis.com/sharing/rest/content/items/{item_id}"

response = requests.get(
    url,
    params={"f": "json"},
    timeout=30
)

print("Status:", response.status_code)

item = response.json()

print("Title:", item.get("title"))
print("Type:", item.get("type"))
print("Access:", item.get("access"))
print("URL:", item.get("url"))
print("Description:", item.get("description"))

Status: 200
Title: Flood Factor risk statistics (v2.0) - Full Dataset
Type: Feature Service
Access: public
URL: https://services7.arcgis.com/iEMmryaM5E3wkdnU/arcgis/rest/services/Flood_Factor_risk_statistics_v2_0_Full_Dataset/FeatureServer
Description: <div>This dataset provides aggregated flood risk statistics at the ZIP Code, County, Congressional District, and State levels from the First Street Foundation Flood Model which is also visualized in the online flood risk assessment tool, <a href='https://floodfactor.com/' target='_blank' rel='nofollow ugc noopener noreferrer'>Flood Factor®</a>.</div><div> </div><div>First Street Foundation is committed to the accuracy of the data and will continue to grow, adapt, and change the Flood Model with new and better information. As such, improvements to the Flood Model are incorporated in quarterly updates. The statistics presented here are consistent with First Street Foundation Flood Model version 2.0 released on April 11, 2022.</div><div> </

In [12]:

service_url = (
    "https://services7.arcgis.com/iEMmryaM5E3wkdnU/"
    "arcgis/rest/services/"
    "Flood_Factor_risk_statistics_v2_0_Full_Dataset/"
    "FeatureServer"
)

response = requests.get(
    service_url,
    params={"f": "json"},
    timeout=30
)

service_info = response.json()

print("SERVICE:", service_info.get("serviceDescription"))
print("AVAILABLE LAYERS:")

for layer in service_info.get("layers", []):
    print(
        "ID:", layer["id"],
        "| Name:", layer["name"]
    )

print("TABLES:")

for table in service_info.get("tables", []):
    print(table)

SERVICE: 
AVAILABLE LAYERS:
ID: 0 | Name: Congressional_Districts_117th
ID: 1 | Name: Counties
ID: 2 | Name: States
ID: 3 | Name: Zip
TABLES:


## Inspecting the layer

In [13]:
layer_url = (
    "https://services7.arcgis.com/iEMmryaM5E3wkdnU/"
    "arcgis/rest/services/"
    "Flood_Factor_risk_statistics_v2_0_Full_Dataset/"
    "FeatureServer/1"
)

response = requests.get(
    layer_url,
    params={"f": "json"},
    timeout=30
)

layer_info = response.json()

print("LAYER:", layer_info.get("name"))
print("DESCRIPTION:", layer_info.get("description"))
print("GEOMETRY TYPE:", layer_info.get("geometryType"))
print("OBJECT ID FIELD:", layer_info.get("objectIdField"))
print("MAX RECORD COUNT:", layer_info.get("maxRecordCount"))
print("ADVANCED QUERY CAPABILITIES:", layer_info.get("advancedQueryCapabilities"))

print("FIELDS:")
for field in layer_info.get("fields", []):
    print(
        field["name"],
        "|", field["type"],
        "|", field.get("alias")
    )

print("EXTENT:")
print(layer_info.get("extent"))

LAYER: Counties
DESCRIPTION: 
GEOMETRY TYPE: esriGeometryPolygon
OBJECT ID FIELD: FID
MAX RECORD COUNT: 2000
ADVANCED QUERY CAPABILITIES: {'supportsPagination': True, 'supportsPaginationOnAggregatedQueries': True, 'supportsQueryRelatedPagination': True, 'supportsQueryWithDistance': True, 'supportsOutDistance': True, 'supportsReturningQueryExtent': True, 'supportsStatistics': True, 'supportsOrderBy': True, 'supportsDistinct': True, 'supportsQueryWithResultType': True, 'supportsSqlExpression': True, 'supportsTimeRelation': True, 'supportsAdvancedQueryRelated': True, 'supportsCountDistinct': True, 'supportsPercentileStatistics': True, 'supportsApproxPercentileStatistics': True, 'supportsSpatialAggregationStatistics': True, 'supportsTrueCurve': True, 'supportedSpatialAggregationStatistics': ['EnvelopeAggregate', 'CentroidAggregate', 'ConvexHullAggregate'], 'supportsLod': True, 'supportsQueryWithLodSR': False, 'supportedLodTypes': ['geohash'], 'supportsReturningGeometryCentroid': True, 'sup

### Found FIPS CODE TO JOIN TO COUNTY SHAPEFILE(Tiger)

In [14]:

query_url = layer_url + "/query"

count_response = requests.get(
    query_url,
    params={
        "where": "1=1",
        "returnCountOnly": "true",
        "f": "json"
    },
    timeout=30
)

count_info = count_response.json()
total_count = count_info.get("count")

print("TOTAL COUNTY RECORDS:", total_count)

sample_response = requests.get(
    query_url,
    params={
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "resultRecordCount": 10,
        "orderByFields": "FID",
        "f": "json"
    },
    timeout=30
)

sample_info = sample_response.json()

if "error" in sample_info:
    print(sample_info["error"])
else:
    sample_df = pd.DataFrame(
        [feature["attributes"] for feature in sample_info["features"]]
    )

    print("SAMPLE RECORDS:")
    display(sample_df)

    print("FIPS FORMAT:")
    print(sample_df["FIPS"].tolist())

TOTAL COUNTY RECORDS: 3220

SAMPLE RECORDS:


,FID,NAME,STATE_NAME,FIPS,count_prop,count_fema,pct_fema_s,count_fs_r,pct_fs_ris,count_fs_1,...,count_fl_2,count_fl_3,count_fl_4,count_fl_5,count_fl_6,count_fl_7,count_fl_8,count_fl_9,Shape__Area,Shape__Length
0,1,Kauai,Hawaii,15007,24932,1883,7.6,1778,7.1,1916,...,376,423,334,740,550,1118,1766,780,1.897992e+09,3.392761e+05
1,2,Honolulu,Hawaii,15003,170442,6778,4.0,9522,5.6,9746,...,4113,5015,3790,7603,5286,8656,9752,3250,1.813625e+09,4.870199e+05
2,3,Hawaii,Hawaii,15001,134501,1836,1.4,25829,19.2,25864,...,3625,4059,3654,4638,6806,12126,15249,3871,1.181673e+10,6.434735e+05
3,4,Maui,Hawaii,15009,50960,1648,3.2,5644,11.1,5761,...,964,1174,890,1600,1561,2906,3835,1861,3.457076e+09,7.287603e+05
4,5,Dillingham,Alaska,02070,2524,0,0.0,27,1.1,30,...,14,8,11,18,9,8,12,18,1.963351e+11,3.417955e+06
5,6,Nome,Alaska,02180,1861,86,4.6,73,3.9,80,...,37,34,22,30,17,22,26,60,3.310824e+11,1.121852e+07
6,7,Kodiak Island,Alaska,02150,8257,0,0.0,335,4.1,317,...,12,3,7,14,14,15,33,272,5.987640e+10,9.559576e+06
7,8,Kenai Peninsula,Alaska,02122,64930,37,0.1,2662,4.1,2718,...,979,1084,899,1433,879,874,1384,1499,1.717064e+11,7.736212e+06
8,9,Matanuska-Susitna,Alaska,02170,106642,33,0.0,3509,3.3,3796,...,1593,1753,1808,2488,1739,1835,2365,1440,2.986805e+11,2.652026e+06
9,10,Anchorage,Alaska,02020,83211,498,0.6,2538,3.1,2816,...,1373,1161,1665,2183,1650,1691,1363,444,1.921338e+10,7.400322e+05



FIPS FORMAT:
['15007', '15003', '15001', '15009', '02070', '02180', '02150', '02122', '02170', '02020']


In [17]:
state_fips = {
    "AL": "01", "AR": "05", "IA": "19", "KY": "21",
    "MS": "28", "MO": "29", "NE": "31", "OK": "40",
    "TN": "47", "WV": "54"
}

query_url = layer_url + "/query"

response = requests.get(
    query_url,
    params={
        "where": "1=1",
        "outFields": "FIPS,NAME,STATE_NAME",
        "returnGeometry": "false",
        "resultOffset": 0,
        "resultRecordCount": 2000,
        "orderByFields": "FID",
        "f": "json"
    },
    timeout=60
)

data = response.json()
first_batch = [f["attributes"] for f in data["features"]]

response = requests.get(
    query_url,
    params={
        "where": "1=1",
        "outFields": "FIPS,NAME,STATE_NAME",
        "returnGeometry": "false",
        "resultOffset": 2000,
        "resultRecordCount": 2000,
        "orderByFields": "FID",
        "f": "json"
    },
    timeout=60
)

data = response.json()
second_batch = [f["attributes"] for f in data["features"]]

fsf_counties = pd.DataFrame(first_batch + second_batch)

fsf_counties["FIPS"] = fsf_counties["FIPS"].astype(str).str.zfill(5)
fsf_counties["STATEFP"] = fsf_counties["FIPS"].str[:2]

# Load TIGER counties
tiger_path = (
    "data/raw/tiger_counties/tl_2025_us_county.shp"
)

counties = gpd.read_file(tiger_path)
counties["GEOID"] = counties["GEOID"].astype(str).str.zfill(5)

study_fips = list(state_fips.values())

fsf_study = fsf_counties[
    fsf_counties["STATEFP"].isin(study_fips)
].copy()

tiger_study = counties[
    counties["STATEFP"].isin(study_fips)
].copy()

fsf_codes = set(fsf_study["FIPS"])
tiger_codes = set(tiger_study["GEOID"])

matched = fsf_codes & tiger_codes
missing_from_fsf = tiger_codes - fsf_codes
not_in_tiger = fsf_codes - tiger_codes

print("FSF study-state counties:", len(fsf_codes))
print("TIGER study-state counties:", len(tiger_codes))
print("Matched counties:", len(matched))
print("Missing from FSF:", len(missing_from_fsf))
print("Not in TIGER:", len(not_in_tiger))

print("MISSING FROM FSF:")
display(
    tiger_study[tiger_study["GEOID"].isin(missing_from_fsf)][
        ["GEOID", "STATEFP", "NAME"]
    ].sort_values("GEOID")
)

print("COUNTIES BY STATE:")
coverage = []

for abbreviation, fips in state_fips.items():
    fsf_state = fsf_study[fsf_study["STATEFP"] == fips]
    tiger_state = tiger_study[tiger_study["STATEFP"] == fips]

    coverage.append({
        "State": abbreviation,
        "FSF Counties": fsf_state["FIPS"].nunique(),
        "TIGER Counties": tiger_state["GEOID"].nunique(),
        "Matched": len(set(fsf_state["FIPS"]) & set(tiger_state["GEOID"]))
    })

display(pd.DataFrame(coverage))

FSF study-state counties: 878
TIGER study-state counties: 878
Matched counties: 878
Missing from FSF: 0
Not in TIGER: 0
MISSING FROM FSF:


,GEOID,STATEFP,NAME


COUNTIES BY STATE:


,State,FSF Counties,TIGER Counties,Matched
0,AL,67,67,67
1,AR,75,75,75
2,IA,99,99,99
3,KY,120,120,120
4,MS,82,82,82
5,MO,115,115,115
6,NE,93,93,93
7,OK,77,77,77
8,TN,95,95,95
9,WV,55,55,55


### Downloading the complete atribute table

In [24]:
state_epsg = {
    "01": 26916,
    "05": 26915,
    "19": 26915,
    "21": 26916,
    "28": 26915,
    "29": 26915,
    "31": 26914,
    "40": 26914,
    "47": 26916,
    "54": 26917
}

state_names = {
    "01": "Alabama",
    "05": "Arkansas",
    "19": "Iowa",
    "21": "Kentucky",
    "28": "Mississippi",
    "29": "Missouri",
    "31": "Nebraska",
    "40": "Oklahoma",
    "47": "Tennessee",
    "54": "West_Virginia"
}

fsf_fields = [
    "count_prop",
    "count_fema",
    "pct_fema_s",
    "count_fs_r",
    "pct_fs_ris",
    "count_fs_1",
    "pct_fs_r_1",
    "count_fs_2",
    "pct_fs_r_2",
    "count_fs_3",
    "pct_fs_r_3",
    "count_fs_4",
    "pct_fs_r_4",
    "count_fs_5",
    "pct_fs_r_5",
    "count_fs_f",
    "pct_fs_fem",
    "avg_risk_s",
    "avg_risk_1",
    "avg_risk_f",
    "avg_risk_2",
    "avg_risk_3",
    "avg_risk_4",
    "count_floo",
    "count_fl_1",
    "count_fl_2",
    "count_fl_3",
    "count_fl_4",
    "count_fl_5",
    "count_fl_6",
    "count_fl_7",
    "count_fl_8",
    "count_fl_9"
]

fsf_state_layers = {}

for state_fips_code, state_name in state_names.items():

    state_gdf = fsf_joined[
        fsf_joined["STATEFP_x"] == state_fips_code
    ].copy()

    state_gdf = state_gdf[
        [
            "GEOID",
            "STATEFP_x",
            "COUNTYFP",
            "NAME_x"
        ] + fsf_fields + ["geometry"]
    ].copy()

    state_gdf = state_gdf.rename(columns={
        "STATEFP_x": "STATEFP",
        "NAME_x": "NAME"
    })

    state_gdf["State"] = state_name

    state_gdf = state_gdf[
        [
            "GEOID",
            "STATEFP",
            "COUNTYFP",
            "NAME"
        ] + fsf_fields + ["State", "geometry"]
    ]

    state_gdf = state_gdf.to_crs(
        epsg=state_epsg[state_fips_code]
    )

    fsf_state_layers[state_name] = state_gdf

    print(
        f"{state_name}: "
        f"{len(state_gdf)} counties | "
        f"{len(state_gdf.columns) - 1} attributes | "
        f"EPSG:{state_epsg[state_fips_code]}"
    )

print("TOTAL COUNTIES:", sum(
    len(gdf) for gdf in fsf_state_layers.values()
))

print("TOTAL ATTRIBUTES:", len(fsf_state_layers["Alabama"].columns) - 1)

print("SAMPLE ATTRIBUTE TABLE:")
display(
    fsf_state_layers["Alabama"].drop(columns="geometry").head()
)

Alabama: 67 counties | 38 attributes | EPSG:26916
Arkansas: 75 counties | 38 attributes | EPSG:26915
Iowa: 99 counties | 38 attributes | EPSG:26915
Kentucky: 120 counties | 38 attributes | EPSG:26916
Mississippi: 82 counties | 38 attributes | EPSG:26915
Missouri: 115 counties | 38 attributes | EPSG:26915
Nebraska: 93 counties | 38 attributes | EPSG:26914
Oklahoma: 77 counties | 38 attributes | EPSG:26914
Tennessee: 95 counties | 38 attributes | EPSG:26916
West_Virginia: 55 counties | 38 attributes | EPSG:26917

TOTAL COUNTIES: 878

TOTAL ATTRIBUTES: 38

SAMPLE ATTRIBUTE TABLE:


,GEOID,STATEFP,COUNTYFP,NAME,count_prop,count_fema,pct_fema_s,count_fs_r,pct_fs_ris,count_fs_1,...,count_fl_1,count_fl_2,count_fl_3,count_fl_4,count_fl_5,count_fl_6,count_fl_7,count_fl_8,count_fl_9,State
3,01075,01,075,Lamar,16317,1644,10.1,1500,9.2,1568,...,34,106,97,125,257,286,587,1137,185,Alabama
15,01025,01,025,Clarke,28293,2049,7.2,2147,7.6,2330,...,74,226,119,228,371,394,791,1801,677,Alabama
19,01123,01,123,Tallapoosa,37029,991,2.7,1814,4.9,1925,...,209,312,221,360,463,517,793,1300,482,Alabama
31,01079,01,079,Lawrence,24024,1428,5.9,921,3.8,971,...,86,264,150,264,360,430,653,740,218,Alabama
45,01059,01,059,Franklin,22540,706,3.1,949,4.2,975,...,38,126,85,122,230,231,413,720,259,Alabama


In [30]:
output_dir = "data/processed/FSF_Flood_Risk_By_State"

os.makedirs(output_dir, exist_ok=True)

existing_gdbs = []

for state_name in fsf_state_layers:
    gdb_path = os.path.join(
        output_dir,
        f"{state_name}_FSF_Flood_Risk.gdb"
    )

    if os.path.exists(gdb_path):
        existing_gdbs.append(gdb_path)

if existing_gdbs:
    print("EXPORT STOPPED: These geodatabases already exist:")
    for path in existing_gdbs:
        print(path)
    raise FileExistsError(
        "No files were exported to avoid overwriting existing data."
    )

for state_name, state_gdf in fsf_state_layers.items():

    gdb_path = os.path.join(
        output_dir,
        f"{state_name}_FSF_Flood_Risk.gdb"
    )

    pyogrio.write_dataframe(
        state_gdf,
        gdb_path,
        layer="FSF_Flood_Risk",
        driver="OpenFileGDB",
        geometry_type="MultiPolygon",
        promote_to_multi=True
    )

    print(
        f"Exported {state_name}: "
        f"{len(state_gdf)} counties"
    )

print("EXPORT COMPLETE")
print("Output directory:", output_dir)

/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_prop of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fema of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fs_r of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fs_1 of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TAR

Exported Alabama: 67 counties
Exported Arkansas: 75 counties
Exported Iowa: 99 counties
Exported Kentucky: 120 counties


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_prop of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fema of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fs_r of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fs_1 of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TAR

Exported Mississippi: 82 counties
Exported Missouri: 115 counties
Exported Nebraska: 93 counties
Exported Oklahoma: 77 counties
Exported Tennessee: 95 counties
Exported West_Virginia: 55 counties
EXPORT COMPLETE
Output directory: data/processed/FSF_Flood_Risk_By_State


/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_prop of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fema of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fs_r of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TARGET_ARCGIS_VERSION=ARCGIS_PRO_3_2_OR_LATER
  ogr_write(
/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/pyogrio/raw.py:723: RuntimeWarning: Field count_fs_1 of type Integer64 will be written as a Float64. To get Integer64, use layer creation option TAR

### ^^ NOw have a folder with gdbs form our 10 states with fsf Data called processed/FSF_Flood_Risk_By_State